<a href="https://colab.research.google.com/github/Chaabmanal2022/Balancing-PR-AUC-and-Explanation-Stability-in-Fraud-Detection/blob/First-branch/Pipeline_1_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **`Environment Setup & Library Imports`**

---



In [1]:
import os
import zipfile
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.base import clone

from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier

import shap

from scipy.stats import spearmanr
from itertools import combinations

warnings.filterwarnings("ignore")

# **`Data Loading & Setup`**

---



In [2]:
# le nom du fichier CSV du dataset PaySim
DATA_PATH = "PS_20174392719_1491204439457_log.csv"
# le nom du fichier compressé téléchargé depuis Kaggle
ZIP_PATH  = "paysim1.zip"

if not os.path.exists(DATA_PATH):
    print("\n[INFO] Téléchargement du dataset via Kaggle API...")

    # Commande sert à télécharger le dataset PaySim depuis Kaggle grâce à l’API Kaggle.
    os.system("kaggle datasets download -d ealaxi/paysim1")
    if os.path.exists(ZIP_PATH):
        # Ouverture et extraction du fichier ZIP
        with zipfile.ZipFile(ZIP_PATH, 'r') as z:
            z.extractall(".")
        print("[INFO] Extraction terminée.")

    # Cette partie s’exécute si le fichier ZIP n’a pas été trouvé après le téléchargement.
    else:
        raise FileNotFoundError(
            "Le fichier ZIP PaySim est introuvable. "
            "Vérifiez votre configuration Kaggle."
        )

df = pd.read_csv(DATA_PATH)


[INFO] Téléchargement du dataset via Kaggle API...
[INFO] Extraction terminée.


# **`FEATURE & TARGET DEFINITION`**

---



In [3]:
CATEGORICAL_FEATURES = ['type']
NUMERICAL_FEATURES   = [
    'step', 'amount',
    'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest'
]
TARGET = 'isFraud'

X = df[CATEGORICAL_FEATURES + NUMERICAL_FEATURES]
y = df[TARGET]

In [4]:
df_sorted = df.sort_values("step").reset_index(drop=True)

# On cherche le step qui correspond au cumul de 80% des lignes,
# pour ne pas couper une transaction au milieu d'un step
cutoff_row = int(len(df_sorted) * 0.8)
cutoff_step = df_sorted.loc[cutoff_row, "step"]

train_df = df_sorted[df_sorted["step"] <= cutoff_step]
test_df  = df_sorted[df_sorted["step"]  > cutoff_step]

X_train, y_train = train_df[CATEGORICAL_FEATURES + NUMERICAL_FEATURES], train_df[TARGET]
X_test,  y_test  = test_df[CATEGORICAL_FEATURES + NUMERICAL_FEATURES],  test_df[TARGET]

print(f"[SPLIT] Step de coupure : {cutoff_step} / {df_sorted['step'].max()}")
print(f"[SPLIT] Train : {len(train_df):,} lignes ({len(train_df)/len(df_sorted)*100:.1f}%) | "
      f"Fraudes : {train_df['isFraud'].sum():,} ({train_df['isFraud'].mean()*100:.4f}%)")
print(f"[SPLIT] Test  : {len(test_df):,} lignes ({len(test_df)/len(df_sorted)*100:.1f}%) | "
      f"Fraudes : {test_df['isFraud'].sum():,} ({test_df['isFraud'].mean()*100:.4f}%)")

[SPLIT] Step de coupure : 355 / 743
[SPLIT] Train : 5,113,884 lignes (80.4%) | Fraudes : 3,963 (0.0775%)
[SPLIT] Test  : 1,248,736 lignes (19.6%) | Fraudes : 4,250 (0.3403%)


# **`Feature Encoding & Standardization`**

---



In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            CATEGORICAL_FEATURES
        ),
        (
            "numerical",
            StandardScaler(),
            NUMERICAL_FEATURES
        )
    ]
)

In [6]:
rows_per_step = int(len(X_train) / cutoff_step)
INNER_CV = TimeSeriesSplit(n_splits=5, gap=rows_per_step)

# Vérification indispensable : s'assurer qu'aucun fold n'a trop peu de fraudes
for fold, (train_idx, val_idx) in enumerate(INNER_CV.split(X_train, y_train), start=1):
    print(f"Fold {fold} | train={len(train_idx):>9,} (fraudes={y_train.iloc[train_idx].sum():>4}) "
          f"| val={len(val_idx):>9,} (fraudes={y_train.iloc[val_idx].sum():>4})")

Fold 1 | train=  837,909 (fraudes= 472) | val=  852,314 (fraudes=1338)
Fold 2 | train=1,690,223 (fraudes=1810) | val=  852,314 (fraudes= 486)
Fold 3 | train=2,542,537 (fraudes=2294) | val=  852,314 (fraudes= 579)
Fold 4 | train=3,394,851 (fraudes=2875) | val=  852,314 (fraudes= 576)
Fold 5 | train=4,247,165 (fraudes=3455) | val=  852,314 (fraudes= 508)


# **`Optuna Hyperparameter Search Space`**

---



**`1. XGBOOST HYPERPARAMETER SEARCH SPACE`**



In [7]:
# Ratio négatif/positif du train, sert de référence pour borner scale_pos_weight
_neg, _pos = (y_train == 0).sum(), (y_train == 1).sum()
_ratio = _neg / _pos
print(f"[INFO] Ratio négatif/positif dans y_train : {_ratio:.1f}")

def get_xgb_params(trial):
    """
    Defines the hyperparameter search space for XGBoost Classifier in Optuna:
    - n_estimators: Number of gradient boosted trees
    - max_depth: Maximum tree depth for base learners
    - learning_rate: Boosting learning rate (shrinkage) on a log scale
    - subsample: Subsample ratio of the training instances
    - gamma: Minimum loss reduction required to make a further partition
    - eval_metric: Precision-Recall AUC optimized for imbalanced data
    """

    params = {

        "n_estimators": trial.suggest_int(
            "n_estimators", 200, 600
        ),

        "max_depth": trial.suggest_int(
            "max_depth", 3, 9
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.2, log=True
        ),

        "subsample": trial.suggest_float(
            "subsample", 0.6, 1.0
        ),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float(
            "gamma", 0.0, 5.0
        ),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 50, _ratio * 1.5, log=True),
        "eval_metric": "aucpr",
        "random_state": 42,
        "n_jobs": -1
    }

    return params

[INFO] Ratio négatif/positif dans y_train : 1289.4


***`2. SHAP Feature Importance Calculation`***

In [8]:
def compute_shap_importance(
    model,
    X_validation,          # matrice prétraitée (numpy)
    y_validation,          # NOUVEAU : les labels de validation
    n_fraud=250,           # nb max de fraudes à expliquer
    n_legit=250,           # nb de transactions légitimes
    random_state=42
):
    """
    Retourne le vecteur d'importance SHAP moyenne (|SHAP|) par feature,
    calculé sur un échantillon STRATIFIÉ : fraudes + légitimes.
    """
    rng = np.random.RandomState(random_state)
    y_arr = np.asarray(y_validation)

    idx_fraud = np.where(y_arr == 1)[0]
    idx_legit = np.where(y_arr == 0)[0]

    # On prend toutes les fraudes disponibles (jusqu'à n_fraud)
    sel_fraud = rng.choice(idx_fraud, size=min(n_fraud, len(idx_fraud)), replace=False)
    sel_legit = rng.choice(idx_legit, size=min(n_legit, len(idx_legit)), replace=False)

    indices = np.concatenate([sel_fraud, sel_legit])
    X_sample = X_validation[indices]

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)

    # Sécurité selon la version de shap (liste [classe0, classe1] ou tableau 3D)
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    elif shap_values.ndim == 3:
        shap_values = shap_values[:, :, 1]

    return np.abs(shap_values).mean(axis=0)

***`3. SPEARMAN SHAP STABILITY`***



In [9]:
def compute_spearman_stability(fold_importances):
    """
    Calculates overall stability of SHAP feature explanations.

    Stability measures the mean pairwise Spearman rank correlation
    between SHAP feature importance vectors across cross-validation folds.
    """

    correlations = []

    # Toutes les combinaisons possibles de deux folds
    for importance_a, importance_b in combinations(
        # fold_importances : Une liste contenant les vecteurs d'importance SHAP calculés pour chaque fold lors de la cv
        fold_importances,
        2
    ):

        # Calculate Spearman rank correlation coefficient
        rho, _ = spearmanr(
            importance_a,
            importance_b
        )

        correlations.append(rho)

    # Compute average correlation across all fold pairs
    stability_score = np.mean(
        correlations
    )

    return stability_score

# ***`Optuna Objective Function`***

---



In [10]:
def objective(trial):

    """
    Evaluates a trial configuration using TimeSeriesSplit Cross-Validation:
    1. Preprocesses training/validation folds dynamically without data leakage.
    3. Trains XGBoost classifier and evaluates PR-AUC on validation fold.
    4. Computes SHAP feature importance per fold.
    5. Returns multi-objective targets: Mean PR-AUC and SHAP Explanation Stability.
    """
    # Sample hyperparameters from defined search spaces
    xgb_params = get_xgb_params(trial)

    # Track metrics and feature importances across cross-validation folds
    fold_pr_auc = []
    fold_shap_importances = []
    fold_details = []

    # ============================================================
    # CROSS-VALIDATION LOOP
    # ============================================================
    trial_num = trial.number + 1

    for fold_num, (train_idx, val_idx) in enumerate(
        INNER_CV.split(X_train, y_train),
        start=1
    ):

        print(f"Execution: Trial {trial_num} | Fold {fold_num}/5")

        # Split raw fold data
        X_tr_raw = X_train.iloc[train_idx]
        X_val_raw = X_train.iloc[val_idx]
        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        # Clone and fit preprocessor on training fold to prevent data leakage
        prep = clone(preprocessor)
        prep.fit( X_tr_raw, y_tr)
        X_tr = prep.transform( X_tr_raw )
        X_val = prep.transform( X_val_raw )

        # # ADASYN resampler with fold-adapted random seed for reproducibility
        # fold_adasyn_params = adasyn_params.copy()
        # fold_adasyn_params["random_state"] = 42 + fold
        # adasyn = ADASYN(**fold_adasyn_params)
        # X_tr_res, y_tr_res = adasyn.fit_resample(X_tr, y_tr)

        # Instantiate and train XGBoost classifier
        model = XGBClassifier(**xgb_params)
        model.fit(X_tr, y_tr)

        # Predict probability for the positive class (fraud)
        y_val_proba = model.predict_proba(
            X_val
        )[:, 1]

        # Compute Precision-Recall AUC for the current fold
        pr_auc = average_precision_score(
            y_val,
            y_val_proba
        )

        fold_pr_auc.append(
            pr_auc
        )

        # Record fold execution metrics
        fold_details.append({
            "fold": fold,
            "pr_auc": float(pr_auc),
            "n_train": int(len(X_tr)),
            "n_validation": int(len(X_val_raw)),
            "fraud_train": int(y_tr.sum()),
            "fraud_validation": int(y_val.sum())
        })

        # Calculate SHAP feature importances on validation set
        shap_importance = compute_shap_importance(
            model=model,
            X_validation=X_val,
            y_validation=y_val.values,
            n_fraud=250,
            n_legit=250,
            random_state=42 + fold_num      # (avant) 42 + fold  ← bug
        )

        fold_shap_importances.append(shap_importance)
        print( f"   PR-AUC = {pr_auc:.6f}" )

    # ============================================================
    # METRIC AGGREGATION & LOGGING
    # ============================================================
    # Calculate average PR-AUC across all folds
    mean_pr_auc = np.mean(fold_pr_auc)

    # Calculate SHAP explanation stability using Spearman correlation
    shap_stability = compute_spearman_stability(fold_shap_importances)

    print( f"\nTrial {trial_num} terminé")

    # Store custom metadata attributes into Optuna trial object
    trial.set_user_attr("fold_pr_auc",[float(x) for x in fold_pr_auc])
    trial.set_user_attr("mean_pr_auc",float(mean_pr_auc))
    trial.set_user_attr("shap_stability",float(shap_stability))
    trial.set_user_attr("fold_details",fold_details)

    print( f"Mean PR-AUC       = {mean_pr_auc:.6f}")
    print( f"SHAP Stability    = {shap_stability:.6f} \n\n")

    # Return multi-objective targets to Optuna
    return (mean_pr_auc,shap_stability)

In [11]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 17.6 MB/s eta 0:00:00


In [ ]:
# import os

# if os.path.exists(DB_PATH):
#     os.remove(DB_PATH)
#     print(f"Base de données {DB_PATH} supprimée avec succès !")
# else:
#     print("Fichier introuvable.")

Base de données /content/drive/MyDrive/pipeline1_baseline.db supprimée avec succès !


In [13]:
import optuna
from optuna.trial import TrialState

# optuna.logging.set_verbosity(optuna.logging.INFO)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ============================================================
# SQLITE SUR GOOGLE DRIVE
# ============================================================

DB_PATH = "/content/drive/MyDrive/pipeline1_baseline.db"

STORAGE = optuna.storages.RDBStorage(
    url=f"sqlite:///{DB_PATH}",

    # Heartbeat toutes les 60 secondes
    heartbeat_interval=60,

    # Si aucun heartbeat pendant 5 minutes,
    # le trial est considéré comme interrompu
    grace_period=300,

    # Évite certaines erreurs de verrouillage SQLite
    engine_kwargs={
        "connect_args": {
            "timeout": 60
        }
    }
)

STUDY_NAME = "XGBoost_PR_AUC_SHAP_Stability_baseline"

# ============================================================
# CREER OU RECHARGER L'ETUDE
# ============================================================

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE,

    directions=[
        "maximize",   # PR-AUC
        "maximize"    # SHAP Stability
    ],

    load_if_exists=True
)

completed_trials = [
    t for t in study.trials
    if t.state == TrialState.COMPLETE
]

print("=" * 70)
print("ETUDE OPTUNA CHARGEE")
print("=" * 70)

print(f"Nom de l'étude       : {STUDY_NAME}")
print(f"Base SQLite          : {DB_PATH}")
print(f"Trials enregistrés   : {len(study.trials)}")
print(f"Trials terminés      : {len(completed_trials)}")

ETUDE OPTUNA CHARGEE
Nom de l'étude       : XGBoost_PR_AUC_SHAP_Stability_baseline
Base SQLite          : /content/drive/MyDrive/pipeline1_baseline.db
Trials enregistrés   : 23
Trials terminés      : 21


In [ ]:
TOTAL_TRIALS = 50

completed_trials = sum(
    t.state == TrialState.COMPLETE
    for t in study.trials
)

remaining_trials = max(
    0,
    TOTAL_TRIALS - completed_trials
)

print(f"Trials terminés actuellement : {completed_trials}/{TOTAL_TRIALS}")
print(f"Trials restants à exécuter   : {remaining_trials}")

if remaining_trials > 0:
    print(f"Lancement de {remaining_trials} trial(s)...\n")

    study.optimize(
        objective,
        n_trials=remaining_trials
    )

else:

    print(f"Les {TOTAL_TRIALS} trials sont déjà terminés.")

Trials terminés actuellement : 21/50
Trials restants à exécuter   : 29
Lancement de 29 trial(s)...

Execution: Trial 24 | Fold 1/5
   PR-AUC = 0.861475
Execution: Trial 24 | Fold 2/5
   PR-AUC = 0.846159
Execution: Trial 24 | Fold 3/5
   PR-AUC = 0.817183
Execution: Trial 24 | Fold 4/5
   PR-AUC = 0.855089
Execution: Trial 24 | Fold 5/5
   PR-AUC = 0.836367

Trial 24 terminé
Mean PR-AUC       = 0.843255
SHAP Stability    = 0.963636 


Execution: Trial 25 | Fold 1/5
   PR-AUC = 0.875763
Execution: Trial 25 | Fold 2/5
   PR-AUC = 0.875016
Execution: Trial 25 | Fold 3/5
   PR-AUC = 0.851362
Execution: Trial 25 | Fold 4/5
   PR-AUC = 0.881449
Execution: Trial 25 | Fold 5/5
   PR-AUC = 0.868275

Trial 25 terminé
Mean PR-AUC       = 0.870373
SHAP Stability    = 0.957273 


Execution: Trial 26 | Fold 1/5
   PR-AUC = 0.856690
Execution: Trial 26 | Fold 2/5
   PR-AUC = 0.855294
Execution: Trial 26 | Fold 3/5
   PR-AUC = 0.831079
Execution: Trial 26 | Fold 4/5
   PR-AUC = 0.866194
Execution: Tri

In [ ]:
from optuna.trial import TrialState

# The variable 'completed_trials' was previously redefined as an integer.
# We need to get the actual list of completed trial objects from the study.
completed_trials_list = [
    t for t in study.trials
    if t.state == TrialState.COMPLETE
]

if completed_trials_list:
    best_trial = max(completed_trials_list, key=lambda t: t.values[0])

    print(f"Meilleur Trial pour PR-AUC : Trial {best_trial.number}")
    print(f"  - PR-AUC max     : {best_trial.values[0]:.6f}")
    print(f"  - SHAP Stability : {best_trial.values[1]:.6f}")
    print(f"  - Hyperparamètres : {best_trial.params}")
else:
    print("Aucun trial complété n'a été trouvé pour déterminer le meilleur.")

# ***`Best Trial Validation & Full Model Test Evaluation`***

---



**`Optimal Parameters & Fold Evaluation`**

---



**`Cross-Validation Metrics Summary`**

---



***`Final Model Training`***

---

